In [ ]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict
from dotenv import load_dotenv
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from langchain_core.messages import HumanMessage, SystemMessage
import os


from typing import TypedDict, Annotated, List
import operator

class JOBState(TypedDict):
    # The specific model to be used by the nodes
    model_name: str 
    
    # Input data
    job_description: str
    user_resume: str
    
    # The final output generated by the AI
    ai_resume: str
    
    # Recommended: track the 'thoughts' or logs of the agent 
    # using an 'operator.add' reducer to keep a history
    steps_taken: Annotated[List[str], operator.add]
    
    # Recommended: score the match percentage between resume and JD
    ats_score: float

chat_model = ChatGroq(
    groq_api_key=os.getenv("GROQ_API_KEY"),
    model_name="llama-3.3-70b-versatile",
    temperature=0.4,
)

from typing import TypedDict, Any
from langchain_core.messages import SystemMessage, HumanMessage

class JOBState(TypedDict):
    model_name: Any  # This stores the initialized model object (e.g., ChatGroq, ChatOpenAI)
    job_description: str
    user_resume: str
    ai_resume: str

def resume_creator(state: JOBState):
    # 1. Access the initialized model from the state
    model = state['model_name']

    # 2. Enhanced System Prompt for better ATS performance
    system_prompt = (
        "You are an expert Resume Architect and Career Coach specializing in ATS (Applicant Tracking Systems). "
        "Your goal is to rewrite resumes to ensure they match high-priority keywords from the job description "
        "while maintaining a professional, clean tone and quantifiable achievements."
    )

    # 3. Structured User Prompt
    user_prompt = f"""
    JOB DESCRIPTION:
    {state['job_description']}

    CURRENT RESUME:
    {state['user_resume']}

    TASK:
    Rewrite the resume to be perfectly tailored for this job. 
    - Use action verbs (e.g., 'Engineered', 'Architected', 'Optimized').
    - Quantify results where possible.
    - Ensure the technical skills section aligns with the requirements.
    - Provide ONLY the rewritten resume content in Markdown format.
    """

    # 4. Standard LangChain message format
    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=user_prompt)
    ]
    
    # 5. Invoke and handle the response object
    # Using .content because .invoke returns a BaseMessage object
    response = model.invoke([system_prompt, HumanMessage(content=user_prompt)])
    return {'ai_resume': response.content}


    graph = StateGraph(JOBState)

graph.add_node('resume_creator',resume_creator)

graph.add_edge(START,'resume_creator')
graph.add_edge('resume_creator',END)

workflow = graph.compile()




In [46]:
import os
from typing import TypedDict, Any
from dotenv import load_dotenv
from langgraph.graph import StateGraph, END
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage

# 1. Load Environment Variables (Make sure GROQ_API_KEY is in your .env)
load_dotenv()

# 2. Define State Schema
class JOBState(TypedDict):
    model_name: Any
    job_description: str
    user_resume: str
    ai_resume: str

# 3. Initialize Groq Model
# Llama-3.3-70b is currently the best balance of speed and resume-writing logic
chat_model = ChatGroq(
    groq_api_key=os.getenv("GROQ_API_KEY"),
    model_name="llama-3.3-70b-versatile",
    temperature=0.4,
)

# 4. Define the Node Logic
def resume_creator(state: JOBState):
    print("---NODE START: Groq is Architecting your Resume---")
    model = state['model_name']
    
    system_prompt = (
        "You are an elite Resume Architect. Rewrite the provided resume to pass ATS filters "
        "and align perfectly with the Job Description. Use professional Markdown formatting."
    )
    
    user_content = f"""
    TAILOR THIS RESUME:
    {state['user_resume']}
    
    BASED ON THIS JD:
    {state['job_description']}
    """
    
    # Groq handles LangChain messages natively
    response = model.invoke([
        SystemMessage(content=system_prompt), 
        HumanMessage(content=user_content)
    ])
    
    return {"ai_resume": response.content}

# 5. Build and Compile
workflow = StateGraph(JOBState)
workflow.add_node("resume_generator", resume_creator)
workflow.set_entry_point("resume_generator")
workflow.add_edge("resume_generator", END)

app = workflow.compile()

# 6. Execute
if __name__ == "__main__":
    test_inputs = {
        "model_name": chat_model, 
        "job_description": "We need a Python developer specializing in FastAPI, LangGraph, and LLM orchestration.", 
        "user_resume": "I am an AI Engineer with experience in Python and building agentic workflows."
    }

    result = app.invoke(test_inputs)
    print("\n--- TAILORED RESUME FROM GROQ ---")
    print(result['ai_resume'])

---NODE START: Groq is Architecting your Resume---

--- TAILORED RESUME FROM GROQ ---
### AI Engineer & Python Developer
#### Contact Information
* Email: [your_email@example.com](mailto:your_email@example.com)
* Phone: 555-555-5555
* LinkedIn: [linkedin.com/in/yourlinkedinprofile](http://linkedin.com/in/yourlinkedinprofile)

### Summary
Highly motivated and experienced AI Engineer with a strong background in Python development, specializing in building scalable and efficient workflows. Proficient in designing and implementing agentic workflows, with a strong interest in adapting to new technologies such as FastAPI, LangGraph, and LLM orchestration.

### Technical Skills
* **Programming Languages:** Python
* **Frameworks:** FastAPI
* **Technologies:** LangGraph, LLM orchestration
* **Agile Methodologies:** Experienced in designing and implementing agentic workflows
* **Operating Systems:** Windows, Linux, macOS

### Professional Experience
#### AI Engineer
* **Company Name**, *City, St